# Notebook 1 — Read & Join the Tables

## Objective

Read the Olist tables from PostgreSQL, inspect their structure and relationships,
aggregate one-to-many tables before joining, and create one ML-ready table
with exactly one row per order.

## Output Artifact

- `ml_orders.parquet`

## Key Rule

One row in the final ML table must represent exactly one order.

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
DB_USER = "olist_user"
DB_PASSWORD = "olist_password"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist"

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database engine created successfully.")

Database engine created successfully.


In [4]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Database connection successful:", result.scalar())

Database connection successful: 1


In [5]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)

tables

,table_name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_translation
7,products
8,sellers


In [6]:
table_counts = {}

for table in tables["table_name"]:
    query = f"SELECT COUNT(*) AS row_count FROM {table}"
    count = pd.read_sql(query, engine).iloc[0]["row_count"]
    table_counts[table] = count

table_counts_df = (
    pd.DataFrame.from_dict(
        table_counts,
        orient="index",
        columns=["row_count"]
    )
    .sort_values("row_count", ascending=False)
)

table_counts_df

,row_count
geolocation,1000163
order_items,112650
order_payments,103886
customers,99441
orders,99441
order_reviews,99224
products,32951
sellers,3095
product_category_translation,71


In [7]:
query = """
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
"""

columns = pd.read_sql(query, engine)

columns

,table_name,column_name,data_type
0,customers,customer_id,character varying
1,customers,customer_unique_id,character varying
2,customers,customer_zip_code_prefix,integer
3,customers,customer_city,character varying
4,customers,customer_state,character varying
5,geolocation,geolocation_zip_code_prefix,integer
6,geolocation,geolocation_lat,double precision
7,geolocation,geolocation_lng,double precision
8,geolocation,geolocation_city,character varying
9,geolocation,geolocation_state,character varying


In [8]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_order_ids
FROM orders;
"""

orders_check = pd.read_sql(query, engine)

orders_check


,total_rows,unique_order_ids
0,99441,99441


In [9]:
query = """
SELECT
    COUNT(*) AS total_items,
    COUNT(DISTINCT order_id) AS orders_with_items,
    MAX(item_count) AS max_items_per_order
FROM (
    SELECT
        order_id,
        COUNT(*) AS item_count
    FROM order_items
    GROUP BY order_id
) AS item_counts;
"""

items_check = pd.read_sql(query, engine)

items_check

,total_items,orders_with_items,max_items_per_order
0,98666,98666,21


In [10]:
query = """
SELECT
    item_count,
    COUNT(*) AS number_of_orders
FROM (
    SELECT
        order_id,
        COUNT(*) AS item_count
    FROM order_items
    GROUP BY order_id
) AS item_counts
GROUP BY item_count
ORDER BY item_count;
"""

item_distribution = pd.read_sql(query, engine)

item_distribution

,item_count,number_of_orders
0,1,88863
1,2,7516
2,3,1322
3,4,505
4,5,204
5,6,198
6,7,22
7,8,8
8,9,3
9,10,8


In [11]:
query = """
SELECT
    order_id,
    COUNT(*) AS item_count,
    SUM(price) AS total_item_price,
    SUM(freight_value) AS total_freight_value,
    AVG(price) AS avg_item_price
FROM order_items
GROUP BY order_id
ORDER BY item_count DESC;
"""

order_items_agg = pd.read_sql(query, engine)

order_items_agg.head(10)

,order_id,item_count,total_item_price,total_freight_value,avg_item_price
0,8272b63d03f5f79c56e9e4120aec44ef,21,31.80,164.37,1.514286
1,ab14fdcfbe524636d65ee38360e22ce8,20,1974.00,288.80,98.700000
2,1b15974a0141d54e36626dca3fdc731a,20,2000.00,202.40,100.000000
3,428a2f660dc84138d969ccd69a0ab6d5,15,982.35,243.30,65.490000
4,9ef13efd6949e4573a18964dd1bbe7f5,15,765.00,18.00,51.000000
5,73c8ab38f07dc94389065f7eba4f297a,14,826.00,188.02,59.000000
6,9bdc4d4c71aa1de4606060929dee888c,14,419.86,108.92,29.990000
7,37ee401157a3a0b28c9c6d0ed8c3b24b,13,389.87,96.07,29.990000
8,af822dacd6f5cff7376413c03a388bb7,12,61.26,182.76,5.105000
9,2c2a19b5703863c908512d135aa6accc,12,248.40,193.32,20.700000


In [12]:
query = """
SELECT
    order_id,
    COUNT(*) AS payment_count,
    SUM(payment_value) AS total_payment_value,
    MAX(payment_installments) AS max_payment_installments
FROM order_payments
GROUP BY order_id
ORDER BY payment_count DESC;
"""

order_payments_agg = pd.read_sql(query, engine)

order_payments_agg.head(10)

,order_id,payment_count,total_payment_value,max_payment_installments
0,fa65dad1b0e818e3ccc5cb0e39231352,29,457.99,1
1,ccf804e764ed5650cd8759557269dc13,26,62.68,1
2,285c2e15bebd4ac83635ccc563dc71f4,22,40.85,1
3,895ab968e7bb0d5659d16cd74cd1650c,21,161.32,1
4,fedcd9f7ccdc8cba3a18defedd1a5547,19,205.74,1
5,ee9ca989fc93ba09a6eddc250ce01742,19,82.73,1
6,4bfcba9e084f46c8e3cb49b0fa6e6159,15,740.76,1
7,21577126c19bf11a0b91592e5844ba78,15,86.99,1
8,3c58bffb70dcf45f12bdf66a3c215905,14,100.57,1
9,4689b1816de42507a7d63a4617383c59,14,529.55,1


In [13]:
query = """
SELECT
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT order_id) AS orders_with_reviews,
    COUNT(DISTINCT review_id) AS unique_reviews
FROM order_reviews;
"""

reviews_check = pd.read_sql(query, engine)

reviews_check

,total_reviews,orders_with_reviews,unique_reviews
0,99224,98673,98410


In [14]:
query = """
SELECT
    order_id,
    COUNT(*) AS review_count,
    AVG(review_score) AS avg_review_score,
    MIN(review_score) AS min_review_score,
    MAX(review_score) AS max_review_score
FROM order_reviews
GROUP BY order_id;
"""

order_reviews_agg = pd.read_sql(query, engine)

order_reviews_agg.head(10)

,order_id,review_count,avg_review_score,min_review_score,max_review_score
0,ec311444a53b23c85b5004ee2682b76b,1,4.0,4,4
1,bc0feb6a96ad7552648f4183cb6ecb63,1,4.0,4,4
2,51122623ee1ccdaadbb6fcd95f077d8a,1,5.0,5,5
3,639b603f09b7050c96e56cd03703437b,1,5.0,5,5
4,c0b4d042df532a3a84b95e89bf2a53c1,1,4.0,4,4
5,b159d0ce7cd881052da94fa165617b05,1,1.0,1,1
6,aff9437f91ba611b2a2fdd890e826d91,1,5.0,5,5
7,da6f4983b13d38e911e7c8f60771ba66,1,5.0,5,5
8,a14c872deca4d99bf8414d3dfb72bc34,1,1.0,1,1
9,969a2e70a6aea318567b09b1b165a52a,1,5.0,5,5


In [15]:
query = """
SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customer_ids
FROM customers;
"""

customers_check = pd.read_sql(query, engine)

customers_check

,total_customers,unique_customer_ids
0,99441,99441


In [16]:
query = """
SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT product_id) AS unique_product_ids
FROM products;
"""

products_check = pd.read_sql(query, engine)

products_check

,total_products,unique_product_ids
0,32951,32951


In [17]:
query = """
SELECT
    COUNT(*) AS total_sellers,
    COUNT(DISTINCT seller_id) AS unique_seller_ids
FROM sellers;
"""

sellers_check = pd.read_sql(query, engine)

sellers_check

,total_sellers,unique_seller_ids
0,3095,3095


In [18]:
print("order_items_agg:", len(order_items_agg), "rows")
print("unique order_ids:", order_items_agg["order_id"].nunique())

print("\norder_payments_agg:", len(order_payments_agg), "rows")
print("unique order_ids:", order_payments_agg["order_id"].nunique())

print("\norder_reviews_agg:", len(order_reviews_agg), "rows")
print("unique order_ids:", order_reviews_agg["order_id"].nunique())

order_items_agg: 98666 rows
unique order_ids: 98666

order_payments_agg: 99440 rows
unique order_ids: 99440

order_reviews_agg: 98673 rows
unique order_ids: 98673


In [19]:
orders = pd.read_sql(
    "SELECT * FROM orders",
    engine
)

print("Orders loaded:", len(orders))

Orders loaded: 99441


In [20]:
ml_orders = orders.copy()

ml_orders = ml_orders.merge(
    order_items_agg,
    on="order_id",
    how="left"
)

ml_orders = ml_orders.merge(
    order_payments_agg,
    on="order_id",
    how="left"
)

ml_orders = ml_orders.merge(
    order_reviews_agg,
    on="order_id",
    how="left"
)

print("Rows:", len(ml_orders))
print("Unique orders:", ml_orders["order_id"].nunique())


Rows: 99441
Unique orders: 99441


In [21]:
customers_data = pd.read_sql(
    """
    SELECT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state
    FROM customers
    """,
    engine
)

print("Customers loaded:", len(customers_data))
print("Unique customer IDs:", customers_data["customer_id"].nunique())

Customers loaded: 99441
Unique customer IDs: 99441


In [22]:
ml_orders = ml_orders.merge(
    customers_data,
    on="customer_id",
    how="left"
)

print("Rows:", len(ml_orders))
print("Unique orders:", ml_orders["order_id"].nunique())

Rows: 99441
Unique orders: 99441


In [23]:
query = """
SELECT
    order_id,
    COUNT(DISTINCT product_id) AS unique_product_count,
    COUNT(DISTINCT seller_id) AS unique_seller_count
FROM order_items
GROUP BY order_id;
"""

order_product_seller_agg = pd.read_sql(query, engine)

order_product_seller_agg.head(10)

,order_id,unique_product_count,unique_seller_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,1
2,000229ec398224ef6ca0657da4fc703e,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1
6,00054e8431b9d7675808bcb819fb4a32,1,1
7,000576fe39319847cbb9d288c5617fa6,1,1
8,0005a1a1728c9d785b8e2b08b904576c,1,1
9,0005f50442cb953dcd1d21e1fb923495,1,1


In [24]:
print(
    "Rows:",
    len(order_product_seller_agg)
)

print(
    "Unique orders:",
    order_product_seller_agg["order_id"].nunique()
)

Rows: 98666
Unique orders: 98666


In [25]:
ml_orders = ml_orders.merge(
    order_product_seller_agg,
    on="order_id",
    how="left"
)

print("Rows:", len(ml_orders))
print("Unique orders:", ml_orders["order_id"].nunique())

Rows: 99441
Unique orders: 99441


In [26]:
query = """
SELECT
    oi.order_id,
    COUNT(DISTINCT p.product_category_name) AS unique_category_count
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY oi.order_id;
"""

order_category_agg = pd.read_sql(query, engine)

order_category_agg.head(10)

,order_id,unique_category_count
0,00010242fe8c5a6d1ba2dd792cb16214,1
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1
3,00024acbcdf0a6daa1e931b038114c75,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,1
6,00054e8431b9d7675808bcb819fb4a32,1
7,000576fe39319847cbb9d288c5617fa6,1
8,0005a1a1728c9d785b8e2b08b904576c,1
9,0005f50442cb953dcd1d21e1fb923495,1


In [27]:
print("Rows:", len(order_category_agg))
print("Unique orders:", order_category_agg["order_id"].nunique())

Rows: 98666
Unique orders: 98666


In [28]:
ml_orders = ml_orders.merge(
    order_category_agg,
    on="order_id",
    how="left"
)

print("Rows:", len(ml_orders))
print("Unique orders:", ml_orders["order_id"].nunique())

Rows: 99441
Unique orders: 99441


In [29]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT geolocation_zip_code_prefix) AS unique_zip_codes
FROM geolocation;
"""

geo_check = pd.read_sql(query, engine)

geo_check

,total_rows,unique_zip_codes
0,1000163,19015


In [30]:
query = """
SELECT
    geolocation_zip_code_prefix,
    AVG(geolocation_lat) AS avg_latitude,
    AVG(geolocation_lng) AS avg_longitude
FROM geolocation
GROUP BY geolocation_zip_code_prefix;
"""

geo_agg = pd.read_sql(query, engine)

print("Rows:", len(geo_agg))
print("Unique ZIP codes:", geo_agg["geolocation_zip_code_prefix"].nunique())

geo_agg.head(10)

Rows: 19015
Unique ZIP codes: 19015


,geolocation_zip_code_prefix,avg_latitude,avg_longitude
0,74710,-16.674842,-49.215224
1,75262,-16.702839,-49.093367
2,69063,-3.115485,-59.996893
3,69906,-9.993372,-67.816433
4,84925,-23.789676,-50.058695
5,76190,-16.806580,-49.922137
6,24416,-22.840098,-43.067783
7,3936,-23.590286,-46.496458
8,91786,-30.186496,-51.143470
9,12502,-22.815724,-45.183558


In [31]:
customers_with_geo = customers_data.merge(
    geo_agg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

print("Rows:", len(customers_with_geo))
print("Unique customer IDs:", customers_with_geo["customer_id"].nunique())

Rows: 99441
Unique customer IDs: 99441


In [32]:
ml_orders = ml_orders.merge(
    customers_with_geo[
        [
            "customer_id",
            "avg_latitude",
            "avg_longitude"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Rows:", len(ml_orders))
print("Unique orders:", ml_orders["order_id"].nunique()) 

Rows: 99441
Unique orders: 99441


In [33]:
print("Shape:", ml_orders.shape)
print("Rows:", len(ml_orders))
print("Unique orders:", ml_orders["order_id"].nunique())

Shape: (99441, 28)
Rows: 99441
Unique orders: 99441


In [34]:
duplicate_orders = ml_orders["order_id"].duplicated().sum()

print("Duplicate order_id rows:", duplicate_orders)

Duplicate order_id rows: 0


In [35]:
print("Columns:")
for i, column in enumerate(ml_orders.columns, start=1):
    print(f"{i}. {column}")

Columns:
1. order_id
2. customer_id
3. order_status
4. order_purchase_timestamp
5. order_approved_at
6. order_delivered_carrier_date
7. order_delivered_customer_date
8. order_estimated_delivery_date
9. item_count
10. total_item_price
11. total_freight_value
12. avg_item_price
13. payment_count
14. total_payment_value
15. max_payment_installments
16. review_count
17. avg_review_score
18. min_review_score
19. max_review_score
20. customer_unique_id
21. customer_zip_code_prefix
22. customer_city
23. customer_state
24. unique_product_count
25. unique_seller_count
26. unique_category_count
27. avg_latitude
28. avg_longitude


In [36]:
missing_values = (
    ml_orders.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
item_count                        775
total_item_price                  775
total_freight_value               775
avg_item_price                    775
unique_category_count             775
unique_seller_count               775
unique_product_count              775
review_count                      768
max_review_score                  768
min_review_score                  768
avg_review_score                  768
avg_latitude                      278
avg_longitude                     278
order_approved_at                 160
total_payment_value                 1
payment_count                       1
max_payment_installments            1
dtype: int64

In [37]:
ml_orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
item_count                              float64
total_item_price                        float64
total_freight_value                     float64
avg_item_price                          float64
payment_count                           float64
total_payment_value                     float64
max_payment_installments                float64
review_count                            float64
avg_review_score                        float64
min_review_score                        float64
max_review_score                        float64
customer_unique_id                       object
customer_zip_code_prefix                

In [38]:
ml_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_item_price,...,max_review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,unique_product_count,unique_seller_count,unique_category_count,avg_latitude,avg_longitude
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,...,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,...,4.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,...,5.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,...,5.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,...,5.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,-23.676370,-46.514627


In [39]:
ml_orders.tail()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_item_price,...,max_review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,unique_product_count,unique_seller_count,unique_category_count,avg_latitude,avg_longitude
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,1.0,72.00,...,5.0,6359f309b166b0196dbf7ad2ac62bb5a,12209,sao jose dos campos,SP,1.0,1.0,1.0,-23.178000,-45.883818
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,1.0,174.90,...,4.0,da62f9e57a76d978d02ab5362c509660,11722,praia grande,SP,1.0,1.0,1.0,-24.001500,-46.449864
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,1.0,205.99,...,5.0,737520a9aad80b3fbbdad19b66b37b30,45920,nova vicosa,BA,1.0,1.0,1.0,-17.898358,-39.373630
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,2.0,359.98,...,2.0,5097a5312c8b157bb7be58ae360ef43c,28685,japuiba,RJ,1.0,1.0,1.0,-22.562825,-42.694574
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03,1.0,68.50,...,5.0,60350aa974b26ff12caad89e55993bd6,83750,lapa,PR,1.0,1.0,1.0,-25.764308,-49.720376


In [40]:
ml_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   item_count                     98666 non-null  float64       
 9   total_item_price               98666 non-null  float64       
 10  total_freight_value            98666 non-null  float64       
 11  avg_item_price 

In [41]:
assert len(ml_orders) == 99441
assert ml_orders["order_id"].nunique() == 99441
assert ml_orders["order_id"].duplicated().sum() == 0

print("All integrity checks passed!")

All integrity checks passed!


In [42]:
first_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

remaining_columns = [
    col for col in ml_orders.columns
    if col not in first_columns
]

ml_orders = ml_orders[first_columns + remaining_columns]

ml_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_item_price,...,max_review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,unique_product_count,unique_seller_count,unique_category_count,avg_latitude,avg_longitude
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,...,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,...,4.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,...,5.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,...,5.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,...,5.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,-23.676370,-46.514627


In [43]:
output_path = "../data/ml_orders.parquet"

ml_orders.to_parquet(
    output_path,
    index=False
)

print(f"Artifact saved successfully: {output_path}")

Artifact saved successfully: ../data/ml_orders.parquet


In [45]:
ml_orders_check = pd.read_parquet(output_path)

print("Shape:", ml_orders_check.shape)
print("Unique orders:", ml_orders_check["order_id"].nunique())

Shape: (99441, 28)
Unique orders: 99441


In [46]:
assert len(ml_orders_check) == 99441
assert ml_orders_check["order_id"].nunique() == 99441
assert ml_orders_check["order_id"].duplicated().sum() == 0

print("✅ ml_orders.parquet is valid")
print("✅ One row = one order")
print("✅ Artifact created successfully")

✅ ml_orders.parquet is valid
✅ One row = one order
✅ Artifact created successfully
